In [79]:
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import pulp
from pulp import LpProblem
import matplotlib.pyplot as plt

I=24
J=15
K=34

n=500000

mois_mapping = {
    'Janvier': 1,
    'Février': 2,
    'Mars': 3,
    'Avril': 4,
    'Mai': 5,
    'Juin': 6,
    'Juillet': 7,
    'Aout': 8,
    'Septembre': 9,
    'Octobre': 10,
    'Novembre': 11,
    'Décembre': 12
}

prices = pd.read_csv('Prices.csv').values
production=pd.read_csv('Production.csv')
production['Mois'] = production['Mois'].map(mois_mapping)
production = np.delete(production.values,2,axis=1)
surface = pd.read_csv('Simulation.csv').values
charges_var= np.delete(pd.read_csv('Charges_var.csv').values,[1,2,3,4,7,8],axis=1)
#print(production)

def get_week(j):
    month=production[j][4]
    year = 2024
    first_day = datetime(year, month, 1)
    if month == 12:
        last_day = datetime(year, 12, 31)
    else:
        last_day = datetime(year, month + 1, 1) - timedelta(days=1)
    weeks = []
    current_day = first_day
    while current_day <= last_day:
        week_num = current_day.isocalendar()[1]
        if week_num == 1 and current_day.month == 12:
            week_num = 53
        if week_num not in weeks:
            week_start = current_day - timedelta(days=current_day.weekday())
            week_end = week_start + timedelta(days=6)
            days_in_current_month = sum(1 for i in range(7) if (week_start + timedelta(days=i)).month == month)
            if days_in_current_month > 3:
                weeks.append(week_num)
        current_day += timedelta(days=7)
    new_weeks=[]
    for week in weeks:
        if week != 48:
            new_weeks.append(week - 14)
    return new_weeks
print(charges_var)

[[1 0 750000]
 [2 0 750000]
 [3 0 750000]
 [4 0 800000]
 [5 0 800000]
 [8 0 750000]
 [9 0 750000]
 [10 0 750000]
 [11 0 1200000]
 [12 0 1200000]
 [13 0 1200000]
 [14 100 100000000]
 [15 0 900000]
 [18 0 800000]
 [19 0 800000]]


In [80]:
C_A = [[[i * j * k for k in range(K)] for j in range(J)] for i in range(I)]
C_V = [[[i * j * k for k in range(K)] for j in range(J)] for i in range(I)]
B = [[[i * j * k for k in range(K)] for j in range(J)] for i in range(I)]
for i in range(I):
    for j in range(J):
        for k in range(K):
            var=0
            for t in range(37):
                # les index ajoutés dans les listes ne sont que pour le parcours des fichiers csv
                p=np.where(prices==production[j][1])[0][0]
                var+=production[j][t+7]*surface[i][2]*prices[p][t+k+production[j][5]]
            C_A[i][j][k]=var
            C_V[i][j][k]=(surface[i][2]*charges_var[j][2])
            B[i][j][k]=C_A[i][j][k]-C_V[i][j][k]

In [89]:
prob = pulp.LpProblem("Max_Profit", pulp.LpMaximize)
X = pulp.LpVariable.dicts("X", (range(I), range(J), range(K)), cat=pulp.LpBinary)

prob += pulp.lpSum(B[i][j][k] * X[i][j][k] for i in range(I) for j in range(J) for k in get_week(j))
for i in range(I):
    prob += pulp.lpSum(X[i]) == 1
    for k in range(K):
        if (i!=21) and (i!=22) and (i!=23):
            if(k in get_week(4)):
                prob += X[i][4][k]==0
            if(k in get_week(3)):
                prob += X[i][3][k]==0
        for j in range(J):
            if(j==9 or j==10 or j==11 or j==13):
                if(k in get_week(j)):
                    if (i!=19) and (i!=20):
                        prob += X[i][j][k]==0
                    if(surface[i][2]>2.87):
                        prob += X[i][j][k]==0 
#prob += pulp.lpSum(production[j][t+7] * X[i][j][k] for t in range(37) for i in range(I) for j in range(J) for k in get_week(j))<=n
prob

Max_Profit:
MAXIMIZE
-313850.0*X_0_0_10 + -188650.0*X_0_0_11 + -70275.0*X_0_0_12 + -375225.0*X_0_0_9 + -65800.0*X_0_10_31 + -61100.0*X_0_10_32 + -37875.0*X_0_10_33 + -100000000.0*X_0_11_26 + -100000000.0*X_0_11_27 + -100000000.0*X_0_11_28 + -100000000.0*X_0_11_29 + -100000000.0*X_0_11_30 + 252350.0*X_0_12_4 + 180325.0*X_0_12_5 + 109475.0*X_0_12_6 + 90200.0*X_0_12_7 + 83125.0*X_0_12_8 + 218675.0*X_0_13_26 + 233700.0*X_0_13_27 + 233075.0*X_0_13_28 + 212750.0*X_0_13_29 + 200775.0*X_0_13_30 + 297625.0*X_0_14_26 + 314050.0*X_0_14_27 + 312550.0*X_0_14_28 + 291075.0*X_0_14_29 + 279275.0*X_0_14_30 + -33850.0*X_0_1_13 + 16225.0*X_0_1_14 + 41625.0*X_0_1_15 + 42275.0*X_0_1_16 + -74800.0*X_0_2_17 + -73550.0*X_0_2_18 + -79125.0*X_0_2_19 + -83575.0*X_0_2_20 + -90875.0*X_0_2_21 + 98350.0*X_0_3_4 + 39575.0*X_0_3_5 + 23250.0*X_0_3_6 + 18050.0*X_0_3_7 + 29300.0*X_0_3_8 + 176125.0*X_0_4_0 + 188075.0*X_0_4_1 + 179800.0*X_0_4_2 + 158150.0*X_0_4_3 + -179500.0*X_0_5_10 + -155100.0*X_0_5_11 + -132200.0*X_0_5_

In [90]:
prob.solve()

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/rachidaitjalloul/anaconda3/lib/python3.10/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/zb/_d8dsvp17n54wwjbqt8lknz80000gn/T/46169f8f195b4c32b17ec587d1b5b7ea-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/zb/_d8dsvp17n54wwjbqt8lknz80000gn/T/46169f8f195b4c32b17ec587d1b5b7ea-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 614 COLUMNS
At line 39504 RHS
At line 40114 BOUNDS
At line 52355 ENDATA
Problem MODEL has 609 rows, 12240 columns and 12825 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 1.44082e+07 - 0.01 seconds
Cgl0002I 585 variables fixed
Cgl0004I processed model has 0 rows, 0 columns (0 integer (0 of which binary)) and 0 elements
Cbc3007W No integer variables - nothing to do
Cuts at root node changed objective from -1.44082e+07 to 

1

In [91]:
print("Statut de la solution:", prob.status)
print("Valeur optimale de la fonction objectif:", prob.objective.value())


Statut de la solution: 1
Valeur optimale de la fonction objectif: 14408172.5


In [15]:
for v in prob.variables():
    if (v.varValue==1):
        print(v.name, "=", v.varValue)
        serre=int(v.name.split("_")[1])+1
        scenario=production[int(v.name.split("_")[2])][0]
        semaine=int(v.name.split("_")[3])+14
        print("Le résultats est dans la serre {}, scenario {}, et semaine {} ".format(serre, scenario,semaine))

X_0_8_33 = 1.0
Le résultats est dans la serre 1, scenario 11, et semaine 47 
X_10_8_33 = 1.0
Le résultats est dans la serre 11, scenario 11, et semaine 47 
X_11_8_33 = 1.0
Le résultats est dans la serre 12, scenario 11, et semaine 47 
X_12_8_33 = 1.0
Le résultats est dans la serre 13, scenario 11, et semaine 47 
X_13_8_33 = 1.0
Le résultats est dans la serre 14, scenario 11, et semaine 47 
X_14_8_33 = 1.0
Le résultats est dans la serre 15, scenario 11, et semaine 47 
X_15_8_33 = 1.0
Le résultats est dans la serre 16, scenario 11, et semaine 47 
X_16_8_33 = 1.0
Le résultats est dans la serre 17, scenario 11, et semaine 47 
X_17_8_33 = 1.0
Le résultats est dans la serre 18, scenario 11, et semaine 47 
X_18_8_33 = 1.0
Le résultats est dans la serre 19, scenario 11, et semaine 47 
X_19_8_33 = 1.0
Le résultats est dans la serre 20, scenario 11, et semaine 47 
X_1_8_33 = 1.0
Le résultats est dans la serre 2, scenario 11, et semaine 47 
X_20_8_33 = 1.0
Le résultats est dans la serre 21, scena